# Finetune Gemma-4 Vision trên Google Colab

Notebook này được tạo để finetune mô hình Gemma-4 (phiên bản E2B hoặc E4B) với dữ liệu hình ảnh (trích xuất thông tin bảng biểu).

**Lưu ý quan trọng trước khi chạy:**
Dữ liệu của bạn hiện đang ở máy Local (`/home/quangnhvn34/dev/me/InsureVN/data/health_insurance/health_insurance_extracted`).
Bạn cần **Upload toàn bộ thư mục `health_insurance_extracted` lên Google Drive** của bạn trước khi chạy notebook này.

In [1]:
# Kết nối Google Colab với Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Cài đặt môi trường Unsloth

In [2]:
%%capture
# Cài đặt Unsloth theo hướng dẫn chính thức
!curl -fsSL https://unsloth.ai/install.sh | sh
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## 2. Load Model & Processor
Chúng ta sẽ sử dụng bản `unsloth/gemma-4-E2B-it` (có khả năng xử lý hình ảnh và phù hợp với GPU 15GB T4 miễn phí trên Colab). Bạn có thể đổi sang bản `E4B` nếu dùng Colab Pro.

In [4]:
from unsloth import FastVisionModel
import torch

# Khởi tạo model và processor
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit = True, # Dùng 4-bit quantization để giảm VRAM
    use_gradient_checkpointing = "unsloth",
)

# Thêm LoRA adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,  # Finetune thêm phần vision
    finetune_language_layers   = True,  
    finetune_attention_modules = True, 
    finetune_mlp_modules       = True, 
    r = 16,                           
    lora_alpha = 16,                  
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

from unsloth import get_chat_template
processor = get_chat_template(
    processor,
    "gemma-4-thinking"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Current model requires 6178 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

## 3. Chuẩn bị Dữ liệu
Script dưới đây sẽ đọc file `oumi_vlm_train.jsonl`, xử lý lại đường dẫn từ Local (`/home/quangnhvn34/...`) sang đường dẫn trên Google Drive, và convert format theo chuẩn yêu cầu của Unsloth Gemma 4.

In [8]:
import json
from datasets import Dataset
import os

# Đường dẫn thư mục chứa data của bạn trên Google Drive Colab
data_dir = "/content/drive/MyDrive/02_AI_Research_&_Data/health_insurance_extracted"
train_file = f"{data_dir}/oumi_vlm_train.jsonl"
eval_file  = f"{data_dir}/oumi_vlm_test.jsonl"

def load_and_format_data(jsonl_file):
    if not os.path.exists(jsonl_file):
        print(f"LỖI: Không tìm thấy {jsonl_file}")
        return None
    
    converted = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            formatted_messages = []
            for msg in data['messages']:
                if msg['role'] == 'user':
                    new_content = []
                    for item in msg['content']:
                        if item['type'] == 'image_path':
                            local_path = item['content']
                            # TỰ ĐỘNG DỊCH ĐƯỜNG DẪN PC SANG GOOGLE DRIVE
                            colab_path = local_path.replace("/home/quangnhvn34/dev/me/InsureVN/data/health_insurance/health_insurance_extracted", data_dir)
                            new_content.append({"type": "image", "image": colab_path})
                        elif item['type'] == 'text':
                            new_content.append({"type": "text", "text": item['content']})
                    formatted_messages.append({"role": "user", "content": new_content})
                elif msg['role'] == 'assistant':
                    ast_text = msg['content']
                    if isinstance(ast_text, str):
                        formatted_messages.append({"role": "assistant", "content": [{"type": "text", "text": ast_text}]})
                    else:
                        formatted_messages.append(msg)
            converted.append({"messages": formatted_messages})
    return Dataset.from_list(converted)

print("Loading Train Dataset...")
hf_train_dataset = load_and_format_data(train_file)
print(f"Train examples: {len(hf_train_dataset)}")

print("\nLoading Eval Dataset...")
hf_eval_dataset = load_and_format_data(eval_file)
print(f"Eval examples: {len(hf_eval_dataset) if hf_eval_dataset else 0}")


Loading Train Dataset...
Train examples: 941

Loading Eval Dataset...
Eval examples: 105


## 4. Cấu hình Training

In [13]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = hf_train_dataset,
    eval_dataset = hf_eval_dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        num_train_epochs = 2, # CHỈNH TẠI ĐÂY: Huấn luyện 2 lần toàn bộ dữ liệu
        learning_rate = 2e-4,
        logging_steps = 1,
        
        # Cấu hình đánh giá
        eval_strategy = "epoch", # Đánh giá sau mỗi epoch để xem sự tiến bộ
        save_strategy = "epoch",
        
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )
)

trainer_stats = trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 941 | Num Epochs = 2 | Total steps = 472
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,859,840 of 5,153,037,856 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.123109,2.213256
2,0.112421,2.195129


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-236/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-472/tokenizer_config.json.


## 5. Lưu mô hình về Google Drive

In [3]:
# Lưu LoRA adapter vào Google Drive để sử dụng sau này
save_dir = "/content/drive/MyDrive/02_AI_Research_&_Data/health_insurance_extracted/gemma4-e2b-finetuned-lora"
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

print(f"Mô hình đã được lưu thành công tại: {save_dir}")

NameError: name 'model' is not defined

## 6. Đánh giá mô hình (Evaluation)
Sau khi train xong, chúng ta có thể load tập test (`oumi_vlm_test.jsonl`) để kiểm tra thử kết quả trực tiếp.

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.6 MB/s eta 0:00:0000:0100:01


In [ ]:
from PIL import Image
from jiwer import cer, wer
import torch

# Chuyển model sang chế độ inference (tắt dropout)
FastVisionModel.for_inference(model)

results = []
num_samples = len(hf_eval_dataset)
print(f"Đang đánh giá {num_samples} mẫu...\n")

for i, sample in enumerate(hf_eval_dataset):
    messages = sample["messages"]

    # Tách user input và ground truth
    user_msg = None
    ground_truth = ""
    for msg in messages:
        if msg["role"] == "user":
            user_msg = msg
        elif msg["role"] == "assistant":
            content = msg["content"]
            if isinstance(content, list):
                ground_truth = " ".join(
                    c["text"] for c in content if c.get("type") == "text"
                )
            elif isinstance(content, str):
                ground_truth = content

    if user_msg is None:
        continue

    # Chuẩn bị input: load ảnh thật từ đường dẫn trên Google Drive
    input_content = []
    skip_sample = False
    for item in user_msg["content"]:
        if item.get("type") == "image":
            try:
                img = Image.open(item["image"]).convert("RGB")
                input_content.append({"type": "image", "image": img})
            except Exception as e:
                print(f"  Lỗi load ảnh mẫu {i}: {e}")
                skip_sample = True
                break
        elif item.get("type") == "text":
            input_content.append({"type": "text", "text": item["text"]})

    if skip_sample:
        continue

    input_messages = [{"role": "user", "content": input_content}]

    try:
        inputs = processor.apply_chat_template(
            input_messages,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
            )

        # Decode chỉ phần generated mới (bỏ phần input prompt)
        generated_ids = output_ids[0][inputs["input_ids"].shape[1] :]
        prediction = processor.decode(
            generated_ids, skip_special_tokens=True
        ).strip()
    except Exception as e:
        print(f"  Lỗi generate mẫu {i}: {e}")
        prediction = ""

    results.append(
        {
            "ground_truth": ground_truth.strip(),
            "prediction": prediction,
        }
    )

    if (i + 1) % 10 == 0:
        print(f"  Đã xử lý {i+1}/{num_samples} mẫu")

# Tính metrics tổng hợp
ground_truths = [r["ground_truth"] for r in results]
predictions = [r["prediction"] for r in results]

total_cer = cer(ground_truths, predictions)
total_wer = wer(ground_truths, predictions)
exact_matches = sum(1 for p, g in zip(predictions, ground_truths) if p == g)
exact_match_rate = exact_matches / len(results) if results else 0

print(f"\n{'='*50}")
print(f"KẾT QUẢ ĐÁNH GIÁ ({len(results)} mẫu)")
print(f"{'='*50}")
print(f"CER (Character Error Rate): {total_cer:.4f}")
print(f"WER (Word Error Rate):      {total_wer:.4f}")
print(f"Exact Match:                {exact_match_rate:.4f} ({exact_matches}/{len(results)})")

# In vài ví dụ so sánh
print(f"\n{'='*50}")
print("VÍ DỤ SO SÁNH (3 mẫu đầu):")
for idx, r in enumerate(results[:3]):
    print(f"\n--- Mẫu {idx+1} ---")
    print(f"EXPECTED: {r['ground_truth'][:300]}")
    print(f"GOT:      {r['prediction'][:300]}")